# 01 — Bronze: Ingest

**Reads:** four MOVER CSV files from the Lakehouse Files section

**Writes:** `bronze.patient_information`, `bronze.patient_history`, `bronze.patient_procedure_events`, `bronze.patient_post_op_complications`

### What this notebook does

This notebook loads the four source CSV files into Bronze Delta tables.

`inferSchema=False` is used so all columns are initially loaded as strings. No cleaning, type conversion, filtering, or row removal is done at this stage. Bronze keeps the source data in its original form for further processing in Silver.

### Checks

- Create the `bronze`, `silver`, and `gold` schemas if they do not already exist.
- Load each CSV and write it as a Bronze Delta table.
- Read each table back after writing and compare the row and column counts with the source DataFrame.

In [1]:
pi = (spark.read
        .option("header", True)
        .option("inferSchema", False)
        .csv("Files/patient_information.csv"))

print("rows=",pi.count())
print("columns=",len(pi.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 3, Finished, Available, Finished, False)

rows= 65728
columns= 23


In [2]:
for s in ["bronze", "silver", "gold"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {s}")

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 4, Finished, Available, Finished, False)

In [3]:
pi.write.mode("overwrite").format("delta").saveAsTable("bronze.patient_information")

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 5, Finished, Available, Finished, False)

In [4]:
b_pi=spark.table("bronze.patient_information")
print("shape:",b_pi.count(),"*",len(b_pi.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 6, Finished, Available, Finished, False)

shape: 65728 * 23


In [5]:
history=(spark.read.option("header",True).option("inferschema",False)
            .csv("Files/patient_history.csv"))
print("shape:",history.count(),len(history.columns))

events=(spark.read.option("header",True).option("inferschema",False)
            .csv("Files/patient_procedure_events.csv"))
print("shape:",events.count(),len(events.columns))

comps=(spark.read.option("header",True).option("inferschema",False)
            .csv("Files/patient_post_op_complications.csv"))
print("shape:",comps.count(),len(comps.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 7, Finished, Available, Finished, False)

shape: 970741 3
shape: 640223 5
shape: 203945 6


In [6]:
history.write.mode("overwrite").format("delta").saveAsTable("bronze.patient_history")
events.write.mode("overwrite").format("delta").saveAsTable("bronze.patient_procedure_events")
comps.write.mode("overwrite").format("delta").saveAsTable("bronze.patient_post_op_complications")

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 8, Finished, Available, Finished, False)

In [7]:
b_history=spark.table("bronze.patient_history")
print("patient_history_shape:",b_history.count(),"*",len(b_history.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 9, Finished, Available, Finished, False)

patient_history_shape: 970741 * 3


In [8]:
b_events=spark.table("bronze.patient_procedure_events")
print("patient_procedure_events_shape:",b_events.count(),"*",len(b_events.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 10, Finished, Available, Finished, False)

patient_procedure_events_shape: 640223 * 5


In [9]:
b_comps=spark.table("bronze.patient_post_op_complications")
print("patient_post_op_complications_shape:",b_comps.count(),"*",len(b_comps.columns))

StatementMeta(, 1f1ebc8c-53b5-420f-9b1e-c1e0f2c69015, 11, Finished, Available, Finished, False)

patient_post_op_complications_shape: 203945 * 6


---

## Bronze layer summary

| table | rows | columns |
|---|---:|---:|
| `bronze.patient_information` | 65,728 | 23 |
| `bronze.patient_history` | 970,741 | 3 |
| `bronze.patient_procedure_events` | 640,223 | 5 |
| `bronze.patient_post_op_complications` | 203,945 | 6 |

The row and column counts were checked before and after writing to Delta. All four tables matched, confirming that no records were lost or duplicated during the write.

### What stays unchanged

The Bronze layer keeps the source data as received. No columns are renamed, cast, trimmed, or filtered at this stage.

### Next

`02_silver_patient_information` uses `bronze.patient_information` to clean and standardize the case-level data, convert data types, validate timestamps, and handle data quality issues.

The other three Bronze tables are retained for later analysis.